# Limpeza dos dados do OpenStreetMap (Overpass API)

**Disciplina:** Coleta, Preparação e Análise de Dados · **Fase 2**

**Entrada:** `dados/processados/osm_pontos_interesse.csv`, gerado por `scripts/coleta_overpass.py`
(escolas, unidades de saúde e abrigos de Porto Alegre).
**Fonte:** © OpenStreetMap contributors, licença ODbL, <https://www.openstreetmap.org/copyright>.

Para cada problema encontrado, o notebook mostra **onde aparece**, **por que é um problema** e **qual decisão foi tomada**,
que é o formato pedido para o relatório `relatorios/limpeza_dados.pdf`. Ao final, um log consolidado
(`osm_log_limpeza.csv`) traz a contagem antes e depois de cada etapa.

**Etapas**
1. Inspeção inicial
2. Duplicatas ponto/polígono
3. Registros sem nome
4. Categorias misturadas
5. Cobertura de abrigos
6. Validações e exportação

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = RAIZ / "dados" / "processados"
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 60)

df = pd.read_csv(PROC / "osm_pontos_interesse.csv")
bruto = df.copy()            # cópia para o comparativo final
log = []                     # uma linha por decisão, vai para o relatório

def registrar(etapa, problema, onde, afetados, antes, depois, decisao, justificativa):
    log.append(dict(etapa=etapa, problema=problema, onde=onde, registros_afetados=int(afetados),
                    total_antes=int(antes), total_depois=int(depois),
                    decisao=decisao, justificativa=justificativa))

print(df.shape)

(593, 10)


## 1. Inspeção inicial

In [2]:
print("Registros por categoria e subtipo (amenity):")
display(df.groupby(["categoria", "amenity"]).size().rename("registros").to_frame())
print("Tipo do objeto no OSM:", df.osm_tipo.value_counts().to_dict())
print("\nNulos por coluna:"); print(df.isna().sum().to_string())
print("\nBounding box (lat/lon):"); print(df[["lat", "lon"]].agg(["min", "max"]).to_string())
print("\nosm_id repetido dentro da mesma categoria:", df.duplicated(["categoria", "osm_tipo", "osm_id"]).sum())

Registros por categoria e subtipo (amenity):


registros
categoria amenity                   
abrigos   social_facility          5
escolas   school                 438
saude     clinic                  98
          hospital                52

Tipo do objeto no OSM: {'way': 466, 'node': 120, 'relation': 7}

Nulos por coluna:
categoria        0
osm_tipo         0
osm_id           0
nome            20
amenity          0
rua            101
numero         133
lat              0
lon              0
coletado_em      0

Bounding box (lat/lon):
           lat        lon
min -30.220404 -51.260296
max -29.967997 -51.050719

osm_id repetido dentro da mesma categoria: 0


**Leitura:**
- Todas as coordenadas estão dentro de Porto Alegre e não há lat/lon nulo.
- Não há `osm_id` repetido, então eventuais duplicatas não são erro de coleta. Seriam objetos diferentes no OSM representando o mesmo lugar.
- O único campo com nulos é `nome`.

## 2. Problema A: o mesmo lugar aparece mais de uma vez

**Onde aparece:** categorias `escolas`, `saude` e `abrigos`.

**Por que é um problema:** no OSM um estabelecimento pode ser mapeado como ponto (`node`) e também como contorno do prédio (`way`),
ou ter vários contornos. Cada objeto vira uma linha, e a contagem de escolas, unidades de saúde e abrigos por bairro ou por nível fica inflada.
Em `abrigos`, que tem pouquíssimos registros, o efeito é proporcionalmente enorme.

**Como detectar:** mesma categoria e mesmo nome normalizado (sem acento e em minúsculas), **e** pelo menos um destes sinais:
- a distância entre os dois é **menor que 500 m**; ou
- o **endereço** (rua + número) é o mesmo, qualquer que seja a distância.

Usar só o nome seria um erro, porque existem unidades diferentes com o mesmo nome em pontos distantes.

**Por que dois critérios:** uma primeira versão usava só a distância (150 m) e deixou passar a *Escola Araguaia*:
um registro é o muro do terreno (`barrier=wall`) e o outro é o prédio (`building=school`), com o **mesmo endereço (nº 780)**,
mas a 443 m um do outro, porque o centro de um terreno grande não coincide com o centro do prédio.
O endereço identifica esse caso mesmo quando a distância não identifica, e o limite foi ampliado para 500 m pelo mesmo motivo.

In [3]:
def haversine_m(lat1, lon1, lat2, lon2):
    r = 6_371_000
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))

df["nome_norm"] = (df.nome.str.lower().str.normalize("NFKD")
                   .str.encode("ascii", "ignore").str.decode("ascii").str.strip())
# endereço só existe quando rua E número estão preenchidos; sem os dois não se usa como critério
df["endereco"] = np.where(df.rua.notna() & df.numero.notna(),
                          df.rua.str.lower().str.strip() + " " + df.numero.astype(str).str.lower().str.strip(), None)
# ao escolher qual registro manter, prefere o polígono: o centro do contorno representa melhor o prédio do que um ponto
PRIORIDADE = {"relation": 0, "way": 0, "node": 1}

def encontrar_duplicatas(d, limite_m, usar_endereco=True):
    """Devolve {indice_removido: (indice_mantido, motivo)}; motivo = distancia | endereco | ambos."""
    d = d.assign(_prio=d.osm_tipo.map(PRIORIDADE))
    remover = {}
    for _, g in d.dropna(subset=["nome_norm"]).groupby(["categoria", "nome_norm"]):
        if len(g) < 2:
            continue
        mantidos = []
        for i, row in g.sort_values("_prio").iterrows():
            ref = None
            for m in mantidos:
                perto = haversine_m(row.lat, row.lon, d.at[m, "lat"], d.at[m, "lon"]) < limite_m
                mesmo_end = usar_endereco and pd.notna(row.endereco) and row.endereco == d.at[m, "endereco"]
                if perto or mesmo_end:
                    ref = (m, "ambos" if perto and mesmo_end else "distancia" if perto else "endereco")
                    break
            if ref is None:
                mantidos.append(i)
            else:
                remover[i] = ref
    return remover

### 2.1 Escolha do limite de distância (análise de sensibilidade)

Em vez de fixar o limite sem justificar, o código conta quantas duplicatas aparecem para cada valor, com e sem o critério de endereço.

In [4]:
sens = pd.DataFrame([{"limite_m": L,
                      "so_distancia": len(encontrar_duplicatas(df, L, usar_endereco=False)),
                      "distancia_ou_endereco": len(encontrar_duplicatas(df, L, usar_endereco=True))}
                     for L in [5, 10, 30, 50, 100, 150, 300, 500, 1000, 2000]])
sens

,limite_m,so_distancia,distancia_ou_endereco
0,5,1,3
1,10,1,3
2,30,6,7
3,50,7,8
4,100,7,8
5,150,7,8
6,300,7,8
7,500,8,8
8,1000,8,8
9,2000,9,9


**Leitura:**
- Só pela distância, o número de duplicatas sobe até cerca de 50 m e depois **parece** estável, mas essa estabilidade esconde a *Escola Araguaia*
  (443 m), que só entra quando o limite passa de 443 m ou quando o endereço é usado.
- Com o critério de endereço o número deixa de depender do limite entre 50 m e 1000 m. É o sinal de que o endereço captura os casos que a distância não captura.
- Acima de 1000 m a distância passa a juntar **unidades diferentes** de mesmo nome (como as unidades da Unimed, que ficam a cerca de 1,8 km uma da outra).
- O limite de **500 m** foi adotado por cobrir terrenos grandes (escolas, hospitais) sem chegar à faixa em que unidades distintas se misturam.

### 2.2 Todos os registros removidos, um a um

In [5]:
LIMITE_M = 500
remover = encontrar_duplicatas(df, LIMITE_M)

casos = pd.DataFrame([{
    "categoria": df.at[i, "categoria"],
    "nome": df.at[i, "nome"],
    "removido": f"{df.at[i, 'osm_tipo']}/{df.at[i, 'osm_id']}",
    "mantido": f"{df.at[m, 'osm_tipo']}/{df.at[m, 'osm_id']}",
    "distancia_m": round(haversine_m(df.at[i, "lat"], df.at[i, "lon"], df.at[m, "lat"], df.at[m, "lon"]), 1),
    "par": f"{df.at[i, 'osm_tipo']}+{df.at[m, 'osm_tipo']}",
    "motivo": motivo,
    "endereco": df.at[i, "endereco"],
} for i, (m, motivo) in remover.items()]).sort_values(["categoria", "nome"]).reset_index(drop=True)

with pd.option_context("display.max_colwidth", None):
    display(casos)
print("\nResumo por tipo de par:"); display(casos.par.value_counts().rename("casos").to_frame())
print("Resumo por categoria:");    display(casos.categoria.value_counts().rename("removidos").to_frame())
print("Resumo por motivo da detecção:"); display(casos.motivo.value_counts().rename("casos").to_frame())

,categoria,nome,removido,mantido,distancia_m,par,motivo,endereco
0,abrigos,Abrigo Cônego Paulo De Nadal,node/5404423950,way/288381201,11.9,node+way,ambos,avenida padre cacique 1372
1,escolas,Colégio Sinodal do Salvador,node/11030466129,way/367324147,23.7,node+way,distancia,NaN
2,escolas,Escola Araguaia,way/466454423,way/231918978,443.0,way+way,ambos,estrada gedeon leite 780
3,escolas,Escola Estadual de 1° Grau Professora Dinah Néri Pereira,node/7223276683,node/6238095094,1.7,node+node,distancia,avenida josé bonifácio 497
4,escolas,Escola Saint Hilaire,way/428647547,way/231949173,45.6,way+way,distancia,rua gervazio braga pinheiro 427
5,escolas,Escola Visconde do Rio Grande,way/427660469,way/243463347,27.8,way+way,distancia,praca aratiba s/nº
6,saude,Unimed,way/420868536,way/420866355,20.7,way+way,distancia,NaN
7,saude,Unimed,node/4117912914,node/4117912915,21.6,node+node,distancia,NaN



Resumo por tipo de par:


,casos
par,
way+way,4
node+way,2
node+node,2


Resumo por categoria:


,removidos
categoria,
escolas,5
saude,2
abrigos,1


Resumo por motivo da detecção:


,casos
motivo,
distancia,6
ambos,2


**Como ler os tipos de par:**
- `node+way`: é um ponto e um contorno do mesmo lugar, que é o caso típico de duplicata de mapeamento.
- `way+way` ou `node+node`: são dois contornos ou dois pontos próximos com o mesmo nome. Podem ser a mesma instalação mapeada duas vezes,
  mas também uma escola com **dois prédios**. Esses são os casos de menor certeza e estão sinalizados na tabela pela coluna `par`.

**Como ler o motivo:** `distancia` = detectado só pela proximidade; `endereco` = detectado só pelo endereço, estando longe
(é o caso da *Escola Araguaia*, que a regra antiga deixava passar); `ambos` = os dois critérios concordam, ou seja, o caso mais seguro.

### 2.3 Contraprova: nomes repetidos que **não** foram removidos

In [6]:
removidos_e_mantidos = set(remover) | {m for m, _ in remover.values()}
rep = df.dropna(subset=["nome_norm"])
rep = rep[rep.duplicated(["categoria", "nome_norm"], keep=False) & ~rep.index.isin(removidos_e_mantidos)]
print(f"{len(rep)} registros ({rep.nome_norm.nunique()} nomes) repetidos mantidos (a mais de {LIMITE_M} m um do outro e sem o mesmo endereço):")
rep.sort_values(["nome", "lat"])[["categoria", "osm_tipo", "osm_id", "nome", "endereco", "lat", "lon"]]

13 registros (7 nomes) repetidos mantidos (a mais de 500 m um do outro e sem o mesmo endereço):


,categoria,osm_tipo,osm_id,nome,endereco,lat,lon
546,saude,way,419339793,Centro Clínico Gaúcho,avenida alberto bins 799,-30.028066,-51.218551
514,saude,way,239664213,Centro Clínico Gaúcho,avenida do forte 171,-30.012187,-51.153793
117,escolas,way,243118489,Escola Afonso Guerreiro Lima,rua guaíba 203,-30.110979,-51.112808
99,escolas,way,235379350,Escola Afonso Guerreiro Lima,rua banco inglês 300,-30.071847,-51.221474
224,escolas,way,317875313,Escola América,rua padre ângelo costa 175,-30.078842,-51.167755
150,escolas,way,244250676,Escola América,NaN,-30.015383,-51.121649
120,escolas,way,243159208,Escola Dolores Alcaraz Caldas,rua doutor carlos niederauer hofmeister 86,-30.151135,-51.137398
190,escolas,way,287753232,Escola Dolores Alcaraz Caldas,rua afonso celso pupe da silveira 25,-30.018831,-51.161337
368,escolas,way,431708682,Escola Porto Alegre,rua joão dallegrave 130,-30.037664,-51.125578
362,escolas,way,430081277,Escola Porto Alegre,rua washington luiz 203,-30.036511,-51.239452


In [7]:
antes = len(df)
df = df.drop(index=list(remover))
registrar("A", "duplicata (mesmo nome e categoria, a menos de 500 m ou com mesmo endereço)",
          "escolas, saude e abrigos; " + ", ".join(f"{k}={v}" for k, v in casos.categoria.value_counts().items()),
          len(remover), antes, len(df),
          f"corrigir: remover o duplicado (< {LIMITE_M} m ou mesmo endereço), mantendo o polígono",
          "é a mesma instalação; contar duas vezes infla o indicador por bairro e por nível")
print(f"{antes} -> {len(df)} registros ({len(remover)} removidos)")
df.groupby("categoria").size().rename("apos_etapa_A").to_frame()

593 -> 585 registros (8 removidos)


,apos_etapa_A
categoria,
abrigos,4
escolas,433
saude,148


**Decisão:** corrigir (remover o duplicado, mantendo o polígono). **Justificativa:** é a mesma instalação física, e contá-la duas vezes distorceria
o indicador de escolas, saúde e abrigos por bairro ou por nível. Os nomes repetidos acima são unidades diferentes (por exemplo, filiais) e foram mantidos.

**Histórico da regra:** a primeira versão usava só distância < 150 m e mantinha as duas linhas da *Escola Araguaia* (muro e prédio do mesmo endereço, a 443 m).
Foi corrigida ao conferir o caso à mão, passando a usar distância < 500 m **ou** mesmo endereço.

**Limitação assumida:** os pares `way+way` podem ser, em alguns casos, dois prédios de uma mesma escola. Se o grupo preferir ser conservador,
basta usar `usar_endereco=False` e reduzir `LIMITE_M` (veja a tabela de sensibilidade) e rodar de novo.

## 3. Problema B: registros sem nome

**Por que é um problema:** sem nome não dá para identificar o local em uma tabela ou no tooltip de um mapa,
e não dá para conferir o ponto em outra fonte.

In [8]:
sem_nome = df[df.nome.isna()].copy()
print("Por categoria e tipo de objeto:")
display(sem_nome.groupby(["categoria", "osm_tipo"]).size().rename("registros").to_frame())
print("Percentual de registros sem nome por categoria:")
display((df.assign(sn=df.nome.isna()).groupby("categoria").sn.mean() * 100).round(1).rename("% sem nome").to_frame())

Por categoria e tipo de objeto:


registros
categoria osm_tipo           
escolas   way              12
saude     node              2
          way               6

Percentual de registros sem nome por categoria:


,% sem nome
categoria,
abrigos,0.0
escolas,2.8
saude,5.4


### 3.1 Os sem nome estão agrupados?

Vários `way` sem nome muito próximos entre si podem ser **prédios de um mesmo complexo escolar** e não escolas distintas.
O código agrupa os sem nome da mesma categoria a menos de 100 m uns dos outros.

In [9]:
def agrupar_proximos(d, raio_m=100):
    """Agrupamento simples por encadeamento: dois pontos a menos de raio_m ficam no mesmo grupo."""
    idx = list(d.index); grupo = {i: i for i in idx}
    def raiz(i):
        while grupo[i] != i:
            grupo[i] = grupo[grupo[i]]; i = grupo[i]
        return i
    for a in range(len(idx)):
        for b in range(a + 1, len(idx)):
            i, j = idx[a], idx[b]
            if haversine_m(d.at[i, "lat"], d.at[i, "lon"], d.at[j, "lat"], d.at[j, "lon"]) < raio_m:
                grupo[raiz(i)] = raiz(j)
    return pd.Series({i: raiz(i) for i in idx})

sem_nome["grupo"] = pd.concat([agrupar_proximos(g) for _, g in sem_nome.groupby("categoria")])
tam = sem_nome.groupby("grupo").size().rename("n_no_grupo")
sem_nome = sem_nome.join(tam, on="grupo")
print(f"{len(sem_nome)} sem nome formam {sem_nome.grupo.nunique()} grupos a menos de 100 m")
display(sem_nome.sort_values(["n_no_grupo", "grupo"], ascending=[False, True])
        [["categoria", "osm_tipo", "osm_id", "lat", "lon", "grupo", "n_no_grupo"]])

20 sem nome formam 15 grupos a menos de 100 m


,categoria,osm_tipo,osm_id,lat,lon,grupo,n_no_grupo
317,escolas,way,425215043,-30.051108,-51.163121,321,5
318,escolas,way,425215193,-30.051402,-51.162681,321,5
319,escolas,way,425215214,-30.051441,-51.162791,321,5
320,escolas,way,425216986,-30.051098,-51.162756,321,5
321,escolas,way,425217265,-30.051195,-51.162755,321,5
481,saude,node,8730543929,-30.043274,-51.153660,482,2
482,saude,node,8730543930,-30.043276,-51.153849,482,2
149,escolas,way,244050597,-30.025459,-51.117782,149,1
193,escolas,way,288987786,-29.992983,-51.192296,193,1
218,escolas,way,317192910,-30.035335,-51.233530,218,1


**Leitura:** grupos com `n_no_grupo > 1` são candidatos a serem **um único estabelecimento com vários prédios**.
Eles **não foram fundidos** automaticamente, porque sem nome não há evidência segura de que sejam o mesmo lugar.
A contagem de escolas pode, portanto, estar um pouco inflada, e isso fica registrado como limitação.

In [10]:
df["sem_nome"] = df.nome.isna()
df["nome"] = df.nome.fillna("(sem nome)")
registrar("B", "registro sem nome", "escolas e saude; ver tabela da etapa 3",
          df.sem_nome.sum(), len(df), len(df),
          "manter e sinalizar: nome='(sem nome)' e coluna sem_nome=True",
          "o ponto tem coordenada e categoria válidas e pode estar em área alagada; descartar subestimaria a contagem")
registrar("B.1", "sem nome agrupados a menos de 100 m (possível mesmo complexo)", "ver etapa 3.1",
          int((sem_nome.n_no_grupo > 1).sum()), len(df), len(df),
          "manter e avisar (não fundir)", "sem nome não há evidência segura de que sejam o mesmo estabelecimento")
int(df.sem_nome.sum())

20

**Decisão:** manter e sinalizar. **Justificativa:** o ponto tem coordenada e categoria válidas, então continua sendo um equipamento que pode
estar em área alagada. Descartá-lo subestimaria a contagem. A coluna `sem_nome` permite filtrar depois.

## 4. Problema C: categorias misturadas

`saude` junta hospitais e clínicas, e `escolas` não distingue o nível de ensino. Em uma enchente, um hospital e uma clínica pequena
não têm o mesmo peso, e creche, escola básica e ensino superior também não.

In [11]:
df = df.rename(columns={"amenity": "subtipo"})
display(df.groupby(["categoria", "subtipo"]).size().rename("registros").to_frame())
registrar("C", "categoria saude mistura hospital e clinic", "saude",
          (df.categoria == "saude").sum(), len(df), len(df),
          "manter a categoria e preservar a coluna subtipo (hospital/clinic)",
          "permite filtrar no dashboard sem refazer a coleta")

registros
categoria subtipo                   
abrigos   social_facility          4
escolas   school                 433
saude     clinic                  96
          hospital                52

**Decisão:** manter a categoria e preservar `subtipo` como coluna.
Os níveis de ensino das escolas **não** são distinguíveis com as tags coletadas, e isso fica registrado como limitação.

## 5. Limitação de cobertura: abrigos

In [12]:
abr = df[df.categoria == "abrigos"]
display(abr[["nome", "osm_tipo", "osm_id", "lat", "lon"]])
registrar("D", "poucos abrigos no OSM (cobertura incompleta)", "abrigos",
          len(abr), len(df), len(df),
          "manter e avisar no dashboard",
          "o OSM não é um cadastro oficial; a análise de abrigos em área alagada não representa a cobertura real")
print(f"{len(abr)} abrigos distintos após a limpeza (eram {(bruto.categoria == 'abrigos').sum()} registros brutos)")

,nome,osm_tipo,osm_id,lat,lon
589,Casa do Migrante Guadalupe,node,8890280357,-30.120935,-51.210165
590,Pensão Nova Vida,way,208171935,-30.048296,-51.178408
591,Abrigo Cônego Paulo De Nadal,way,288381201,-30.071739,-51.237489
592,Albergue Municipal,way,419342137,-30.019781,-51.211207


4 abrigos distintos após a limpeza (eram 5 registros brutos)


**Decisão:** manter e avisar. Restam poucos abrigos, porque o OSM não é um cadastro oficial.
A análise de abrigos em área alagada **não deve ser lida como cobertura completa**, e isso vai constar como limitação no dashboard.

## 6. Validações finais e exportação

In [13]:
final = df.drop(columns="nome_norm").reset_index(drop=True)

checagens = {
    "sem coordenada nula": final[["lat", "lon"]].notna().all().all(),
    "sem osm_id repetido por categoria": not final.duplicated(["categoria", "osm_tipo", "osm_id"]).any(),
    "nenhum nome nulo": final.nome.notna().all(),
    "sem_nome coerente com '(sem nome)'": (final.sem_nome == (final.nome == "(sem nome)")).all(),
    "dentro da caixa de Porto Alegre": final.lat.between(-30.3, -29.9).all() and final.lon.between(-51.4, -51.0).all(),
    "só registros removidos na etapa A": len(bruto) - len(final) == len(remover),
}
for nome, ok in checagens.items():
    print("OK  " if ok else "FALHOU", nome)
assert all(checagens.values())

OK   sem coordenada nula
OK   sem osm_id repetido por categoria
OK   nenhum nome nulo
OK   sem_nome coerente com '(sem nome)'
OK   dentro da caixa de Porto Alegre
OK   só registros removidos na etapa A


In [14]:
resumo = pd.DataFrame({
    "bruto": bruto.groupby("categoria").size(),
    "limpo": final.groupby("categoria").size(),
})
resumo["removidos"] = resumo.bruto - resumo.limpo
resumo["sem_nome"] = final.groupby("categoria").sem_nome.sum()
resumo.loc["TOTAL"] = resumo.sum()
resumo

,bruto,limpo,removidos,sem_nome
categoria,,,,
abrigos,5,4,1,0
escolas,438,433,5,12
saude,150,148,2,8
TOTAL,593,585,8,20


In [15]:
final.to_csv(PROC / "osm_pontos_interesse_limpo.csv", index=False, encoding="utf-8")
log_df = pd.DataFrame(log)
log_df.to_csv(PROC / "osm_log_limpeza.csv", index=False, encoding="utf-8")
sens.to_csv(PROC / "osm_sensibilidade_limite.csv", index=False, encoding="utf-8")
casos.to_csv(PROC / "osm_duplicatas_removidas.csv", index=False, encoding="utf-8")
with pd.option_context("display.max_colwidth", None):
    display(log_df[["etapa", "problema", "registros_afetados", "total_antes", "total_depois", "decisao"]])

,etapa,problema,registros_afetados,total_antes,total_depois,decisao
0,A,"duplicata (mesmo nome e categoria, a menos de 500 m ou com mesmo endereço)",8,593,585,"corrigir: remover o duplicado (< 500 m ou mesmo endereço), mantendo o polígono"
1,B,registro sem nome,20,585,585,manter e sinalizar: nome='(sem nome)' e coluna sem_nome=True
2,B.1,sem nome agrupados a menos de 100 m (possível mesmo complexo),7,585,585,manter e avisar (não fundir)
3,C,categoria saude mistura hospital e clinic,148,585,585,manter a categoria e preservar a coluna subtipo (hospital/clinic)
4,D,poucos abrigos no OSM (cobertura incompleta),4,585,585,manter e avisar no dashboard


**Arquivos gerados** em `dados/processados/`:
- `osm_pontos_interesse_limpo.csv`: base limpa usada pelo dashboard.
- `osm_log_limpeza.csv`: uma linha por decisão (problema, onde, afetados, antes/depois, decisão e justificativa).
- `osm_duplicatas_removidas.csv`: os registros removidos, um a um, com distância e tipo de par.
- `osm_sensibilidade_limite.csv`: duplicatas encontradas por limite de distância, com e sem o critério de endereço.

**Pendente:** associar cada ponto a um bairro com *spatial join* usando a malha oficial do ObservaPOA,
atenção à grafia dos nomes (acentos e apóstrofos).